In [2]:
from dotenv import load_dotenv
import os
load_dotenv('.env')

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3,
    base_url="https://api.openai.com/v1",
    api_key=os.getenv("OPENAI_KEY")
)

weekLLM = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.3,
    base_url="https://api.openai.com/v1",
    api_key=os.getenv("OPENAI_KEY")
)

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model='text-embedding-3-large',
    # model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_KEY"),
    base_url="https://api.openai.com/v1"
)


In [3]:
inputText = ''
with open("inputs/yaskawa-l1000a.txt", "r" , encoding="utf-8") as f:
  inputText = f.read()

from langchain_core.documents import Document

inputDoc = Document(page_content=inputText)

from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

chunks = splitter.split_text(inputText)

print(len(chunks))

1353


In [4]:
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Vector Store
vStore = Chroma(
    embedding_function=embeddings,
    persist_directory="./ChromaDB/db"
)

# Dense Retriever (MMR)
dense_retriever = vStore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 15,
        "fetch_k": 50,
        "lambda_mult": 0.3
    }
)

# Sparse Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 15

# Hybrid Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

C:\Users\kave240ad\AppData\Local\Temp\ipykernel_12928\3770077070.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [5]:
from langchain_core.tools.retriever import create_retriever_tool

retriver_tool_yaskawa_l1000a =create_retriever_tool(
    retriever = hybrid_retriever,
    name="retriver_yaskawa_L1000A",
    description="search in vector store db and bm25_retriever about YASKAWA AC Drive L1000A AC Drive for Elevator Applications"
)


tools=[retriver_tool_yaskawa_l1000a]

In [6]:
from langgraph.graph import START,END,StateGraph
from typing import Annotated,Sequence
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.prompts import PromptTemplate

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage],add_messages]
    
def agent( state:AgentState):
    """
    Invokes the agent model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply end.

    Args:
        state (messages): The current state

    Returns:
        dict: The updated state with the agent response appended to messages
    """
    messages = state['messages']
    
    llm_with_tool = llm.bind_tools(tools=tools)
    response = llm_with_tool.invoke(messages)
    
    return {"messages":[response]}


In [7]:

from pydantic import BaseModel,Field
from typing import Literal
def grade_document(state:AgentState)-> Literal["generate", "rewrite"]:
    
    #print("---- GREAD DOCUMENTS CALL ---")
    class grade(BaseModel):
        binary_score : str = Field(description="Relevance score 'yes' or 'no' ")
        
    llm_with_structured_output = weekLLM.with_structured_output(grade)
    
    prompt = PromptTemplate(
        template="""You are a grader assessing the relevance of a retrieved document to a user question.
        Here is the retrieved document:
        {context}

        Here is the user question:
        {question}

        Decide whether the document is useful for answering the question.

        The document does NOT need to directly answer the question.
        If it contains related concepts, partial information, background knowledge, or anything that could help answer the question, consider it relevant.

        Be generous in your judgment — if there is any meaningful semantic connection, mark it as relevant.

        Respond with a binary score:
        'yes' if relevant or potentially useful
        'no' if completely unrelated
        """,
        input_variables=["context", "question"],
    )

    
    chain = prompt | llm_with_structured_output
    
    messages = state['messages']
    
    lastMess = messages[-1].content
    question = messages[0].content
    
    resault = chain.invoke({"context":lastMess,"question":question})
    score = resault.binary_score

    if score == "yes":
        print("---DECISION: RELEVANT---")
        return "generate"

    else:
        print("---DECISION: DOCS NOT RELEVANT---")
        #print(score)
        return "rewrite"


In [8]:
from langchain_core.output_parsers import StrOutputParser

from langchain_core.messages import HumanMessage,AIMessage
def generate(state:AgentState):
    
    messages = state['messages']
    
    question = messages[0].content
    docs = messages[-1].content
    
    prompt = PromptTemplate(
        template="""
        You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
        Question: {question} 
        Context: {context} 
        Answer:
        """,
        input_variables=['question','context']
    )
    
    chain = prompt | llm | StrOutputParser()
    
    response = chain.invoke({"question":question,"context":docs})
    
    return {"messages":[ AIMessage( content=response )]}
    

In [9]:

from langchain_core.messages import HumanMessage,AIMessage
def rewrite(state:AgentState):
    
    print("---TRANSFORM QUERY---")
    messages = state["messages"]
    question = messages[0].content

    prompt = PromptTemplate(
        template=""" 
        Look at the input and try to reason about the underlying semantic intent / meaning. \n 
        Here is the initial question:
        {question} 
        Formulate an improved question for search in vector dataBase(RAG) only retuen the new query
        try to be creative and new query be little diffrent to question:
        """,
        input_variables=['question']
    )
    
    chain = prompt | weekLLM | StrOutputParser()

    help = """The previous query i created for the vector database did not return suitable results. 
    Here is a suggested query that might work better; i can use it and call my tool again whit a new query : """
    # Grader
    response = chain.invoke({'question':question})
    return {
        "messages": [
            AIMessage(content=help+response)
        ]
    }

In [10]:
from langgraph.graph import START,END,StateGraph
from langgraph.prebuilt import ToolNode,tools_condition

workFlow = StateGraph(AgentState)

workFlow.add_node("agent",agent)
workFlow.add_node("retriver",ToolNode(tools=tools))
workFlow.add_node("generate",generate)
workFlow.add_node("rewrite",rewrite)

workFlow.add_edge(START,"agent")

workFlow.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools":"retriver",
        END: END
    }
)

workFlow.add_conditional_edges(
    "retriver",grade_document
)

workFlow.add_edge("generate", END)
workFlow.add_edge("rewrite", "agent")

graph = workFlow.compile()

print( graph.get_graph().draw_ascii() )

# from IPython.display import Image, display
# display(Image(graph.get_graph(xray=True).draw_mermaid_png()))


                    +-----------+                
                    | __start__ |                
                    +-----------+                
                           *                     
                           *                     
                           *                     
                      +-------+                  
                      | agent |.                 
                   ...+-------+ ....             
                ...        *        ....         
            ....           *            ...      
          ..               *               ....  
+----------+               *                   ..
| retriver |               *                    .
+----------+....           *                    .
      .         ...        *                    .
      .            ....    *                    .
      .                ..  *                    .
+----------+          +---------+              ..
| generate |          | rewrite |          ....  


In [12]:
from langchain_core.messages import AIMessage,ToolMessage


for event in graph.stream(
    {"messages": "how to set Brake Close Delay  in yaskawa l1000a "},
    stream_mode="tasks"
):

    pass
    print("\n" + "-"*40)
    
    task = {}
    if 'name' in event:
        # print(f"Task Name: {event['name']}")
        task['name'] = event['name']
    if 'input' in event:
        # print("Status: Getting Input...")
        task['input'] = "Getting Input..."
        
    if 'result' in event and event['result']:
        messages = event['result'].get('messages', [])
        
        for msg in messages:
            
            if isinstance(msg, AIMessage):
                
                if msg.tool_calls:
                    # print("🛠 Tool Call Detected:")
                    toolsCall = []
                    for tool in msg.tool_calls:
                        toolObj = {}
                        toolObj['name'] = tool['name']
                        toolObj['args'] = tool['args']
                        # print(f"  - Tool: {tool['name']}")
                        # print(f"  - Args: {tool['args']}")
                        toolsCall.append( toolObj )
                    if len( toolsCall ) :
                        task['tools'] = toolsCall
                else :
                    task['ai_resault'] = msg.content     
            elif isinstance(msg , ToolMessage):
                task['tool_result'] = "tool create resault"
    print(task)




----------------------------------------
{'name': 'agent', 'input': 'Getting Input...'}

----------------------------------------
{'name': 'agent', 'tools': [{'name': 'retriver_yaskawa_L1000A', 'args': {'query': 'Brake Close Delay setting in Yaskawa L1000A'}}]}

----------------------------------------
{'name': 'retriver', 'input': 'Getting Input...'}
---DECISION: RELEVANT---

----------------------------------------
{'name': 'retriver', 'tool_result': 'tool create resault'}

----------------------------------------
{'name': 'generate', 'input': 'Getting Input...'}

----------------------------------------
{'name': 'generate', 'ai_resault': 'To set the Brake Close Delay in the Yaskawa L1000A, adjust parameter S1-07 (Brake Close Delay Time). This parameter determines the delay time after reaching zero speed before the brake control output (H2- = 50) is reset to apply the brake. The setting range is from 0.00 seconds up to the value of S1-05, with a default of 0.10 seconds.'}


In [ ]:
data = {'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 81, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DcnzOemwjFASmPqs5Cb79G97gzfrW', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e017c-9b35-7f32-a9fa-610c00bb85ff-0', tool_calls=[{'name': 'retriver_vector_hd5l_vfd_drive', 'args': {'query': 'Slip compensation gain HD5L'}, 'id': 'call_c9Df9FUXxrtyB1tt8PB4NwxL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 28, 'total_tokens': 109, 'input_token_details': {}, 'output_token_details': {'reasoning': 0}})]}}
key = list(data.keys())
data[key[0]]['messages'][0].tool_calls

agent = next(iter(data.values()))
msg = agent["messages"][0]
tool_calls = msg.tool_calls
tool_calls

[{'name': 'retriver_vector_hd5l_vfd_drive',
  'args': {'query': 'Slip compensation gain HD5L'},
  'id': 'call_c9Df9FUXxrtyB1tt8PB4NwxL',
  'type': 'tool_call'}]